**Data required:** Run `06_swap_simulations/run_simulations.py` first to populate `06_swap_simulations/output/` with simulation pkl files. Precomputed summary CSVs are already in `summary_stats/` if you only need the outputs.

In [1]:
import pickle
import numpy as np
import pandas as pd

In [2]:
from pyprojroot import here

import importlib.util

_spec = importlib.util.spec_from_file_location("opinion_functions", here() / "src" / "opinion_functions.py")
_mod = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(_mod)
fun = _mod

In [3]:
# Parameters
data_folder = here() / "06_swap_simulations" / "output"
homophily_values = [0.0, 0.25, 0.5, 0.75, 1]
m_values = [2, 5]
num_agents_values = [1000]
num_sim = 1000

simulation_results = fun.load_simulation_results_with_swaps(data_folder, homophily_values, m_values, num_agents_values, num_sim)

Files in output folder:
homophily_0.5_m_5_num_agents_1000_sim_1000_min_swapped.pkl
homophily_0.25_m_2_num_agents_1000_sim_1000_maj.pkl
homophily_0.5_m_5_num_agents_1000_sim_1000_maj_swapped.pkl
homophily_0.0_m_2_num_agents_1000_sim_1000_min.pkl
homophily_0.0_m_5_num_agents_1000_sim_1000_min_swapped.pkl
homophily_0.5_m_5_num_agents_1000_sim_1000_maj.pkl
homophily_0.0_m_5_num_agents_1000_sim_1000_maj_swapped.pkl
homophily_1_m_2_num_agents_1000_sim_1000_min.pkl
homophily_1_m_5_num_agents_1000_sim_1000_min.pkl
homophily_0.0_m_2_num_agents_1000_sim_1000_maj_swapped.pkl
homophily_0.5_m_2_num_agents_1000_sim_1000_maj.pkl
homophily_0.75_m_5_num_agents_1000_sim_1000_maj_swapped.pkl
homophily_0.0_m_2_num_agents_1000_sim_1000_min_swapped.pkl
homophily_0.75_m_5_num_agents_1000_sim_1000_min_swapped.pkl
homophily_0.0_m_5_num_agents_1000_sim_1000_min.pkl
homophily_0.25_m_5_num_agents_1000_sim_1000_maj.pkl
homophily_0.5_m_2_num_agents_1000_sim_1000_maj_swapped.pkl
homophily_0.75_m_5_num_agents_1000_si

In [4]:
# Keyed by (homophily, m, num_agents), where m=2 and num_agents=1000
m = 2
num_agents = 1000
homophilyvec = [0.0, 0.25, 0.5, 0.75, 1]
minority_fraction = 0.333
majority_fraction = 1 - minority_fraction
num_sim = 1000 # Number of simulations
# Load results

In [5]:
# Initialize dictionaries to store results and their statistics for each key
mean_majority_opinion_stats = {}
mean_minority_opinion_stats = {}
mean_overall_opinion_stats = {}

# Iterate over all keys in simulation_results
for result_key in simulation_results.keys():
    homophily, m, num_agents, swap_idx = result_key

    # Extract simulation data for the current swap
    majority_data = simulation_results[result_key]["majority"]
    minority_data = simulation_results[result_key]["minority"]

    # Compute mean opinions for the current swap
    mean_majority = np.mean(majority_data, axis=1)  # Mean for each simulation
    mean_minority = np.mean(minority_data, axis=1)  # Mean for each simulation
    mean_overall = minority_fraction * mean_minority + majority_fraction * mean_majority

    # Compute statistics for the current swap
    mean_majority_opinion_stats[result_key] = {
        "mean": np.mean(mean_majority),
        "std": np.std(mean_majority),
        "min": np.min(mean_majority),
        "max": np.max(mean_majority),
    }

    mean_minority_opinion_stats[result_key] = {
        "mean": np.mean(mean_minority),
        "std": np.std(mean_minority),
        "min": np.min(mean_minority),
        "max": np.max(mean_minority),
    }

    mean_overall_opinion_stats[result_key] = {
        "mean": np.mean(mean_overall),
        "std": np.std(mean_overall),
        "min": np.min(mean_overall),
        "max": np.max(mean_overall),
    }

# Example of accessing the statistics for a specific swap
# Example: mean_majority_opinion_stats[(homophily, m, num_agents, swap_idx)]["min"]


In [6]:
# Specify the swaps you want to include
swaps_to_include = [10, 20, 50, 100, 200, 300]

# Helper function to add rows for each result_key
def add_to_multiindex_data(result_key, stats_dicts):
    homophily, m, num_agents, swap_idx = result_key  # Unpack the updated key

    # Ensure only keys for the current swap are included
    if swap_idx not in swaps_to_include:
        return  # Skip this result_key if it does not match the current swap
    
    row = {
        "homophily": homophily  # Add homophily to the row
    }
    # Add statistics for each opinion type
    for opinion_type, stats_dict in stats_dicts.items():
        row[(opinion_type, "mean")] = np.round(stats_dict["mean"], 2)
        row[(opinion_type, "std")] = np.round(stats_dict["std"], 2)
        row[(opinion_type, "min")] = np.round(stats_dict["min"], 2)
        row[(opinion_type, "max")] = np.round(stats_dict["max"], 2)
    multiindex_data.append(row)

# Process data and generate separate tables for each swap
for swap in swaps_to_include:
    multiindex_data = []  # Clear data for this swap
    unique_keys = list({(key[0], key[3]): key for key in mean_majority_opinion_stats.keys()}.values())
    for result_key in unique_keys:
        if result_key[3] == swap:  # Ensure swap matches
            stats_dicts = {
                "majority": mean_majority_opinion_stats[result_key],
                "minority": mean_minority_opinion_stats[result_key],
                "overall": mean_overall_opinion_stats[result_key],
            }
            add_to_multiindex_data(result_key, stats_dicts)
    
    # Create the DataFrame
    multiindex_df = pd.DataFrame(multiindex_data)
    multiindex_df = multiindex_df.set_index(['homophily'])  # Only include 'homophily' in the index

    # Set the MultiIndex columns
    multiindex_df.columns = pd.MultiIndex.from_tuples(multiindex_df.columns)

    # Construct the file name
    m_value, num_agents_value = m, num_agents  # Use appropriate values from the last processed key
    file_name = f"summary_stats/summary_statistics_m{m_value}_agents{num_agents_value}_swap_{swap}.csv"

    # Save the DataFrame to the file
    multiindex_df.to_csv(file_name)
    print(f"DataFrame saved for swap={swap}: {file_name}")


DataFrame saved for swap=10: summary_stats/summary_statistics_m5_agents1000_swap_10.csv
DataFrame saved for swap=20: summary_stats/summary_statistics_m5_agents1000_swap_20.csv
DataFrame saved for swap=50: summary_stats/summary_statistics_m5_agents1000_swap_50.csv
DataFrame saved for swap=100: summary_stats/summary_statistics_m5_agents1000_swap_100.csv
DataFrame saved for swap=200: summary_stats/summary_statistics_m5_agents1000_swap_200.csv
DataFrame saved for swap=300: summary_stats/summary_statistics_m5_agents1000_swap_300.csv


In [7]:
multiindex_df

majority                     minority                     overall  \
              mean   std    min    max     mean   std    min    max    mean   
homophily                                                                     
0.00         23.64  1.02  20.24  26.69    41.38  0.62  39.54  43.28   29.55   
0.25         28.44  0.92  25.00  31.49    44.82  0.57  43.13  46.42   33.90   
0.50         30.85  1.06  28.15  34.36    45.73  0.45  44.23  47.34   35.80   
0.75         30.63  1.07  27.56  34.55    45.74  0.46  44.21  47.20   35.66   
1.00         31.27  1.08  27.73  34.98    45.82  0.43  44.50  47.26   36.11   

                               
            std    min    max  
homophily                      
0.00       0.59  27.45  31.43  
0.25       0.59  31.64  35.91  
0.50       0.65  34.16  37.99  
0.75       0.65  33.73  38.05  
1.00       0.65  34.23  38.28